# Train a LEGO part detector for Brick Sorter

This notebook teaches a YOLO model to draw a box around **every LEGO part in a photo**, so Brick Sorter can handle cluttered piles and touching parts. It only finds parts; Brickognize still identifies them.

It trains on the *Tagged images with LEGO bricks* dataset from Gdańsk University of Technology (2,933 real photos with hand-drawn boxes plus 2,908 renders, CC BY 4.0).

**Before you start**
1. In Colab: **Runtime → Change runtime type → T4 GPU**.
2. Download the dataset zip (step 2 below) and put it in a Google Drive folder called `brick-detector`.
3. Run the cells in order. Training takes roughly 1–3 hours on a free T4. Progress is saved to Drive, so if Colab disconnects you can resume.

At the end you get `lego-detector.pt` in your Drive, which you use with `python brick_sorter.py scan photos/ --model lego-detector.pt`.

## 1. Set up

In [ ]:
%pip install -q "ultralytics>=8.3,<8.4" onnx onnxslim

import os, glob, zipfile, shutil, random
from pathlib import Path
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (training will be very slow; switch the runtime to T4 GPU)")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content/drive/MyDrive/brick-detector")
    LOCAL = Path("/content")
except ImportError:                      # running outside Colab
    WORK = Path("brick-detector").resolve()
    LOCAL = Path(".").resolve()
WORK.mkdir(parents=True, exist_ok=True)
print("Working folder:", WORK)

## 2. Get the dataset

The dataset site blocks automated downloads, so download it in your browser:

1. Open **https://doi.org/10.34808/anq4-rn44** (*Tagged images with LEGO bricks*) and download the zip.
2. Upload it to your Google Drive folder **`brick-detector`** (the folder created by the cell above).

**Optional, more data:** *Tagged images with LEGO bricks part 2* (**https://doi.org/10.34808/7kk9-tn08**) adds 15,608 photos on a white desk. Its boxes were drawn automatically by a model and never cleaned, so some are wrong. Add it later if your detector misses parts on plain backgrounds; leave it out for the first run.

Then run the next cell. It unzips every zip in the folder (including zips inside zips) to the Colab machine's fast local disk.

In [ ]:
RAW = LOCAL / "raw"

zips = sorted(WORK.glob("*.zip"))
if not zips:
    raise SystemExit(f"No .zip files in {WORK}. Upload the dataset zip there (see step 2), then run this cell again.")

if RAW.exists():
    shutil.rmtree(RAW)
RAW.mkdir(parents=True)
for z in zips:
    print("Unzipping", z.name, f"({z.stat().st_size / 1e9:.2f} GB)")
    with zipfile.ZipFile(z) as f:
        f.extractall(RAW / z.stem)

# Some archives contain more zips
while True:
    inner = [p for p in RAW.rglob("*.zip")]
    if not inner:
        break
    for z in inner:
        print("Unzipping inner", z.relative_to(RAW))
        with zipfile.ZipFile(z) as f:
            f.extractall(z.with_suffix(""))
        z.unlink()

n_xml = sum(1 for _ in RAW.rglob("*.xml"))
n_img = sum(1 for p in RAW.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print(f"Found {n_img} images and {n_xml} annotation files")
for d in sorted({p.relative_to(RAW).parts[:3] for p in RAW.rglob("*.xml")})[:15]:
    print("  ", "/".join(d))

## 3. Settings

The defaults suit a first run on a free T4. Bigger `MODEL_SIZE` or `IMGSZ` is more accurate but slower.

In [ ]:
MODEL_SIZE = "s"        # n (fastest), s (good default), m (slower, better)
IMGSZ = 960             # training image size; larger helps with small parts in wide shots
EPOCHS = 40             # stops early if it stops improving
BATCH = -1              # -1 picks the largest batch that fits the GPU
INCLUDE_RENDERS = True  # the rendered images help, but photos matter most
VAL_FRACTION = 0.12     # share of images held back to measure accuracy
RUN_NAME = f"lego-{MODEL_SIZE}-{IMGSZ}"

## 4. Convert the annotations

The dataset uses PASCAL VOC XML boxes. This converts them into YOLO format with a single class, `brick`, since the detector only needs to find parts. Images are shrunk to at most 1280 px on the long side to make training faster; the boxes are scaled with them.

In [ ]:
import random, shutil, xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path
from PIL import Image, ImageOps

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_images(root):
    """Index every image under root by lower-case file name and by stem."""
    by_name, by_stem = {}, {}
    for p in Path(root).rglob("*"):
        if p.suffix.lower() in IMG_EXTS:
            by_name.setdefault(p.name.lower(), p)
            by_stem.setdefault((str(p.parent), p.stem.lower()), p)
    return by_name, by_stem

def read_voc(xml_path):
    root = ET.parse(xml_path).getroot()
    size = root.find("size")
    w = int(float(size.findtext("width", "0"))) if size is not None else 0
    h = int(float(size.findtext("height", "0"))) if size is not None else 0
    boxes, labels = [], []
    for obj in root.iter("object"):
        bb = obj.find("bndbox")
        if bb is None:
            continue
        x0, y0, x1, y1 = (float(bb.findtext(k, "0")) for k in ("xmin", "ymin", "xmax", "ymax"))
        boxes.append((x0, y0, x1, y1))
        labels.append((obj.findtext("name") or "").strip())
    return root.findtext("filename") or "", w, h, boxes, labels

def open_matching(img_path, xml_w, xml_h):
    """Open the image the way the annotator saw it.

    labelImg sizes are written from the pixels it displayed. Phone photos often
    carry an EXIF rotation, so try both the raw and the EXIF-rotated image and
    keep whichever matches the XML size. If neither matches, scale the boxes.
    """
    raw = Image.open(img_path)
    raw.load()
    candidates = [raw, ImageOps.exif_transpose(raw)]
    for im in candidates:
        if xml_w and xml_h and im.size == (xml_w, xml_h):
            return im.convert("RGB"), 1.0, 1.0
    im = candidates[1]
    sx = im.size[0] / xml_w if xml_w else 1.0
    sy = im.size[1] / xml_h if xml_h else 1.0
    return im.convert("RGB"), sx, sy

def convert_voc_to_yolo(src_roots, out_dir, max_side=1280, val_frac=0.12,
                        include_renders=True, seed=0):
    """Turn PASCAL VOC folders into a one-class ('brick') YOLO dataset."""
    out = Path(out_dir)
    if out.exists():
        shutil.rmtree(out)
    for split in ("train", "val"):
        (out / "images" / split).mkdir(parents=True)
        (out / "labels" / split).mkdir(parents=True)

    items = []
    for root in src_roots:
        by_name, by_stem = find_images(root)
        for xml in Path(root).rglob("*.xml"):
            if not include_renders and "render" in str(xml.relative_to(root)).lower():
                continue
            try:
                fname, w, h, boxes, labels = read_voc(xml)
            except ET.ParseError:
                continue
            img = (by_stem.get((str(xml.parent), xml.stem.lower()))
                   or by_name.get(Path(fname).name.lower()))
            if img is None:
                continue
            items.append((xml, img, w, h, boxes, labels))

    random.Random(seed).shuffle(items)
    n_val = max(1, int(len(items) * val_frac)) if len(items) > 1 else 0
    stats = Counter()
    label_names = Counter()
    for i, (xml, img, w, h, boxes, labels) in enumerate(items):
        split = "val" if i < n_val else "train"
        try:
            im, sx, sy = open_matching(img, w, h)
        except Exception:
            stats["unreadable"] += 1
            continue
        W, H = im.size
        lines = []
        for (x0, y0, x1, y1), lab in zip(boxes, labels):
            x0, x1 = sorted((x0 * sx, x1 * sx))
            y0, y1 = sorted((y0 * sy, y1 * sy))
            x0, y0, x1, y1 = max(0, x0), max(0, y0), min(W, x1), min(H, y1)
            if x1 - x0 < 2 or y1 - y0 < 2:
                stats["tiny_box_skipped"] += 1
                continue
            label_names[lab] += 1
            lines.append(f"0 {(x0 + x1) / 2 / W:.6f} {(y0 + y1) / 2 / H:.6f} {(x1 - x0) / W:.6f} {(y1 - y0) / H:.6f}")
        k = max_side / max(W, H)
        if k < 1:
            im = im.resize((round(W * k), round(H * k)), Image.LANCZOS)
        name = f"{i:06d}_{img.stem}"[:120]
        im.save(out / "images" / split / f"{name}.jpg", quality=92)
        (out / "labels" / split / f"{name}.txt").write_text("\n".join(lines))
        stats[split] += 1
        stats["boxes"] += len(lines)
        if not lines:
            stats["no_bricks"] += 1
    (out / "data.yaml").write_text(
        f"path: {out.resolve()}\ntrain: images/train\nval: images/val\nnames:\n  0: brick\n")
    return stats, label_names


DATA = LOCAL / "yolo_lego"
stats, labels = convert_voc_to_yolo([RAW], DATA, max_side=1280, val_frac=VAL_FRACTION,
                                    include_renders=INCLUDE_RENDERS)
print(dict(stats))
print("Label names found in the XML files (all treated as 'brick'):", dict(labels.most_common(10)))
if stats["train"] == 0:
    raise SystemExit("No annotated images were converted. Check that the zip contains .xml files next to the images.")

Check a few converted images. Every part should have a box around it; if boxes are shifted or rotated, the conversion went wrong.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

train_imgs = sorted((DATA / "images" / "train").glob("*.jpg"))
samples = random.sample(train_imgs, min(6, len(train_imgs)))
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, p in zip(axes.ravel(), samples):
    im = Image.open(p).convert("RGB")
    d = ImageDraw.Draw(im)
    W, H = im.size
    for line in (DATA / "labels" / "train" / (p.stem + ".txt")).read_text().splitlines():
        _, cx, cy, w, h = map(float, line.split())
        d.rectangle([(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H], outline=(255, 190, 0), width=max(2, W // 300))
    ax.imshow(im); ax.set_title(p.name[:30], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Train

Starts from a model pre-trained on everyday photos. Results and checkpoints are written to `brick-detector/runs/` in your Drive after every epoch.

Augmentation includes vertical flips and small rotations, since parts photographed from above have no fixed "up".

In [ ]:
from ultralytics import YOLO

model = YOLO(f"yolo11{MODEL_SIZE}.pt")
results = model.train(
    data=str(DATA / "data.yaml"),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    project=str(WORK / "runs"), name=RUN_NAME, exist_ok=True,
    patience=10, cos_lr=True, close_mosaic=5,
    degrees=15, flipud=0.5, fliplr=0.5, scale=0.5,
    workers=2, plots=True,
)

**If Colab disconnected during training:** reconnect, run step 1 (set up) and step 3 (settings), then run the cell below. It picks up from the last saved epoch. (Step 2 and 4 also need re-running first, because the converted images were on the Colab machine's local disk.)

In [ ]:
last = WORK / "runs" / RUN_NAME / "weights" / "last.pt"
if not last.exists():
    print("No checkpoint yet at", last)
else:
    try:
        model = YOLO(str(last))
        results = model.train(resume=True)
    except AssertionError:
        print("Training already finished, nothing to resume. Carry on to step 6.")

## 6. Check accuracy

`mAP50` is the main number: the share of parts found with a reasonably tight box. Above 0.9 is very good for this task. `Recall` tells you how many parts it finds at all.

In [ ]:
best = WORK / "runs" / RUN_NAME / "weights" / "best.pt"
model = YOLO(str(best))
m = model.val(data=str(DATA / "data.yaml"), imgsz=IMGSZ, plots=False)
print(f"mAP50: {m.box.map50:.3f}   mAP50-95: {m.box.map:.3f}   precision: {m.box.mp:.3f}   recall: {m.box.mr:.3f}")

val_all = sorted((DATA / "images" / "val").glob("*.jpg"))
val_imgs = random.sample(val_all, min(6, len(val_all)))
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, r in zip(axes.ravel(), model.predict([str(p) for p in val_imgs], imgsz=IMGSZ, conf=0.3, verbose=False)):
    ax.imshow(r.plot()[:, :, ::-1]); ax.set_title(f"{len(r.boxes)} parts", fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Try it on your own pile

Upload one or more photos of your own parts. `CONF` is how sure the model must be to count something: lower it if parts are missed, raise it if it boxes things that aren't parts.

In [ ]:
CONF = 0.3

try:
    from google.colab import files
    uploaded = list(files.upload().keys())
except ImportError:
    uploaded = []   # outside Colab: put file paths here
for name in uploaded:
    r = model.predict(name, imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    plt.figure(figsize=(12, 9)); plt.imshow(r.plot()[:, :, ::-1]); plt.axis("off")
    plt.title(f"{name}: {len(r.boxes)} parts found"); plt.show()

## 8. Save the model

Copies the best model to `brick-detector/lego-detector.pt` in your Drive, plus an ONNX version (`lego-detector.onnx`) that could later run inside the web app.

Download `lego-detector.pt` from Drive, put it next to `brick_sorter.py`, then:

```
pip install ultralytics
python brick_sorter.py scan photos/ --model lego-detector.pt --debug
```

In [ ]:
out_pt = WORK / "lego-detector.pt"
shutil.copy(best, out_pt)
print("Saved:", out_pt)
try:
    onnx_path = Path(YOLO(str(best)).export(format="onnx", imgsz=IMGSZ, simplify=True))
    shutil.copy(onnx_path, WORK / "lego-detector.onnx")
    print("Saved:", WORK / "lego-detector.onnx")
except Exception as e:
    print("ONNX export skipped:", e)
print(f"Use the same image size when detecting: --imgsz {IMGSZ}")

---
Dataset: Boiński, T. et al., *Photos and rendered images of LEGO bricks*, Scientific Data (2023), CC BY 4.0. Credit the authors if you share the model.